# Railway AI Inspector - Model Evaluation

## Objective

Evaluate the trained CNN model on unseen test images and calculate:

- Test Accuracy
- Predictions
- Confusion Matrix
- Classification Report

In [1]:
# ==================================================
# IMPORT LIBRARIES
# ==================================================

from pathlib import Path

import torch
import torch.nn as nn

from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print("Libraries Imported Successfully")

Libraries Imported Successfully


## Device Configuration

In [2]:
# ==================================================
# DEVICE CONFIGURATION
# ==================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


## Dataset Paths

In [3]:
# ==================================================
# DATASET PATHS
# ==================================================

PROJECT_ROOT = Path.cwd().parent

TEST_DIR = PROJECT_ROOT / "data" / "test"
MODEL_PATH = PROJECT_ROOT / "models" / "railway_cnn_baseline.pth"

print("Test Path :", TEST_DIR)
print("Model Path:", MODEL_PATH)

Test Path : d:\Railway_AI_Inspector\data\test
Model Path: d:\Railway_AI_Inspector\models\railway_cnn_baseline.pth


## Load Test Dataset

In [4]:
# ==================================================
# TEST TRANSFORMATIONS
# ==================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Test Transform Created Successfully")

Test Transform Created Successfully


In [5]:
# ==================================================
# LOAD TEST DATASET
# ==================================================

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=transform
)

print("Test Samples:", len(test_dataset))

print("\nClasses:")
print(test_dataset.classes)

print("\nClass Mapping:")
print(test_dataset.class_to_idx)

Test Samples: 15

Classes:
['broken_rail', 'crack', 'misalignment', 'normal', 'surface_wear']

Class Mapping:
{'broken_rail': 0, 'crack': 1, 'misalignment': 2, 'normal': 3, 'surface_wear': 4}


In [6]:
# ==================================================
# TEST DATALOADER
# ==================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print("Test DataLoader Created Successfully")

Test DataLoader Created Successfully


## CNN Architecture

In [7]:
# ==================================================
# BASELINE CNN ARCHITECTURE
# ==================================================

class RailwayCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 56 * 56,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                5
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


print("CNN Architecture Created Successfully")

CNN Architecture Created Successfully


In [16]:
import os

print(os.listdir(PROJECT_ROOT / "models"))

['railway_cnn_baseline.pth']


In [17]:
print(MODEL_PATH)
print(MODEL_PATH.exists())

d:\Railway_AI_Inspector\models\railway_cnn_baseline.pth
True


In [20]:
model = RailwayCNN().to(device)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


## Test Evaluation

In [21]:
# ==================================================
# TEST EVALUATION
# ==================================================

all_labels = []
all_predictions = []

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 26.67%


In [22]:
# ==================================================
# CONFUSION MATRIX
# ==================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[2 0 0 1 0]
 [1 0 0 2 0]
 [0 0 0 3 0]
 [1 0 0 2 0]
 [0 0 0 3 0]]


In [23]:
# ==================================================
# CLASSIFICATION REPORT
# ==================================================

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=test_dataset.classes
    )
)

              precision    recall  f1-score   support

 broken_rail       0.50      0.67      0.57         3
       crack       0.00      0.00      0.00         3
misalignment       0.00      0.00      0.00         3
      normal       0.18      0.67      0.29         3
surface_wear       0.00      0.00      0.00         3

    accuracy                           0.27        15
   macro avg       0.14      0.27      0.17        15
weighted avg       0.14      0.27      0.17        15



d:\Railway_AI_Inspector\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Railway_AI_Inspector\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Railway_AI_Inspector\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
